# 09 -- Boosting models (XGBoost, LightGBM, CatBoost, GradientBoosting)

Gradient boosting usually gives the best PD discrimination on tabular
credit data. This notebook:

1. Plots true per-iteration train/validation **loss curves** for all four
   libraries (early-stopping view).
2. Tunes each with `GridSearchCV` (compact, realistic grids).
3. Compares the four on the untouched holdout and inspects importance.

`scale_pos_weight` handles the ~8% default rate. Grids are intentionally
small; widen them or lower `SEARCH_FRAC` runtime permitting.

In [ ]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.modeling import (
    classification_metrics,
    cv_results_frame,
    evaluate_split,
    load_modeling_data,
    make_pipeline,
    metrics_frame,
    run_grid_search,
    subsample,
    train_valid_report,
)
from src.modeling import plots

data = load_modeling_data()
pd.Series(data.summary())


In [ ]:
# Fraction of the TRAIN partition used for GridSearchCV. The holdout is never
# touched during tuning. Lower this (e.g. 0.3) for faster iteration, then set
# back to 1.0 to confirm the final model on the full training data.
SEARCH_FRAC = 0.3
search = subsample(data, frac=SEARCH_FRAC)
pd.Series(search.summary())


## Per-iteration loss curves

Each model is fit once on a train/validation split with log loss recorded
every boosting round. The green line marks the best (minimum validation)
iteration - where early stopping would halt.

In [ ]:
from src.modeling.boosting import (
    fit_catboost_curve,
    fit_gbm_curve,
    fit_lgbm_curve,
    fit_xgb_curve,
)

Xtr, ytr = search.X_train, search.y_train
Xva, yva = data.X_holdout, data.y_holdout

curves = {
    "XGBoost": fit_xgb_curve(Xtr, ytr, Xva, yva, num_boost_round=500),
    "LightGBM": fit_lgbm_curve(Xtr, ytr, Xva, yva, num_boost_round=500),
    "CatBoost": fit_catboost_curve(Xtr, ytr, Xva, yva, num_boost_round=500),
    "GradientBoosting": fit_gbm_curve(Xtr, ytr, Xva, yva, num_boost_round=300),
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, (name, lc) in zip(axes.ravel(), curves.items()):
    plots.plot_loss_curve(lc.train_loss, lc.valid_loss, ax=ax, title=f"{name} loss curve")
plt.tight_layout()
plt.show()

pd.Series({k: v.best_iteration for k, v in curves.items()}, name="best_iteration")

## XGBoost tuning

In [ ]:
from xgboost import XGBClassifier

pos_weight = search.pos_weight
xgb = XGBClassifier(
    n_estimators=400, tree_method="hist", eval_metric="logloss",
    scale_pos_weight=pos_weight, random_state=42, n_jobs=-1,
)
xgb_grid = {
    "max_depth": [3, 4, 6],
    "learning_rate": [0.03, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8],
    "reg_lambda": [1.0, 5.0],
}
gs_xgb = run_grid_search(xgb, xgb_grid, search, refit="roc_auc", n_jobs=1)
cv_results_frame(gs_xgb).head(8)

## LightGBM tuning

In [ ]:
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier(
    n_estimators=500, scale_pos_weight=pos_weight,
    random_state=42, n_jobs=-1, verbose=-1,
)
lgbm_grid = {
    "num_leaves": [31, 63],
    "learning_rate": [0.03, 0.1],
    "subsample": [0.8],
    "subsample_freq": [1],
    "colsample_bytree": [0.8],
    "reg_lambda": [1.0, 5.0],
}
gs_lgbm = run_grid_search(lgbm, lgbm_grid, search, refit="roc_auc", n_jobs=1)
cv_results_frame(gs_lgbm).head(8)

## CatBoost tuning

In [ ]:
from catboost import CatBoostClassifier

cat = CatBoostClassifier(
    iterations=400, scale_pos_weight=pos_weight, loss_function="Logloss",
    random_seed=42, allow_writing_files=False, verbose=False,
)
cat_grid = {
    "depth": [4, 6],
    "learning_rate": [0.03, 0.1],
    "l2_leaf_reg": [3.0, 7.0],
}
gs_cat = run_grid_search(cat, cat_grid, search, refit="roc_auc", n_jobs=1)
cv_results_frame(gs_cat).head(8)

## Model comparison on holdout

In [ ]:
best_models = {
    "xgboost": gs_xgb.best_estimator_,
    "lightgbm": gs_lgbm.best_estimator_,
    "catboost": gs_cat.best_estimator_,
}
rows = {name: evaluate_split(m, data.X_holdout, data.y_holdout) for name, m in best_models.items()}
comparison = metrics_frame(rows).round(4).sort_values("roc_auc", ascending=False)
comparison

In [ ]:
# Overlay ROC / PR for the tuned boosters
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, m in best_models.items():
    proba = m.predict_proba(data.X_holdout)[:, 1]
    plots.plot_roc_pr(data.y_holdout.to_numpy(), proba, label=name, axes=axes)
plt.tight_layout()
plt.show()

## Feature importance (best booster)

In [ ]:
best_name = comparison.index[0]
gs = best_models[best_name]  # reuse the diagnostics helpers below
pre = gs.named_steps["preprocess"]
feat_names = list(pre.transform(search.X_train.iloc[:1]).columns)
model = gs.named_steps["model"]
importances = getattr(model, "feature_importances_", None)
if importances is not None:
    fig, ax = plt.subplots(figsize=(8, 8))
    plots.plot_feature_importance(feat_names, importances, top=20, ax=ax, title=f"{best_name} importance")
    plt.show()
print("best booster on holdout:", best_name)

## Takeaways

- Loss curves show each library converging; the validation minimum is the
  early-stopping point and guards against overfitting.
- The comparison table ranks tuned boosters by holdout AUC/KS - this
  selects the Stage-5 champion to port into `src/train.py` for Stage 6.
- Reported metrics in `reports/metrics.json` must come from the runnable
  pipeline, not these exploratory cells.